# YOLO 특강 1회차 실습 — 알약 탐지로 현업 흐름 한 바퀴

**파일**: `yolo_day1_pill_lab_v1.0.ipynb` · **환경**: Google Colab (T4 GPU) · **모델**: YOLO26n · **데이터**: Ultralytics Medical Pills

| 장 | 내용 | 슬라이드 |
|---|---|---|
| 1 | 설치와 환경 확인 | — |
| 2 | 데이터셋 내려받기, YAML과 폴더 규칙 | 18 |
| 3 | 라벨 한 줄 읽기, 픽셀 변환, 라벨 통계 | 17, 24, 12 |
| 4 | IoU 손계산 | 19 |
| 5 | NMS 손계산 | 20 |
| 6 | AP 손계산 | 22 |
| 7 | 사전학습 모델을 그대로 돌려 보기 | 25 |
| 8 | 전이학습 (증강 기본값, 학습 배치 보기) | 25–27 |
| 9 | 학습 곡선 읽기 | 28 |
| 10 | 평가: mAP50과 mAP50-95 | 23, 29 |
| 11 | 추론: 신뢰도 기준, 두 개의 출력 머리 | 21, 29 |
| 12 | 오류 분석: 사진별 TP·FP·FN | 30 |
| 13 | 확장: 처방 세트로 보정하기 | 35 |
| 14 | 과제: 내 사진 20장으로 직접 해 보기 | 37 |

> 시작 전에 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 셀은 위에서 아래로 순서대로 실행합니다.

## 1장. 설치와 환경 확인

In [ ]:
# Colab에는 ultralytics가 설치되어 있지 않습니다. 1~2분 걸립니다.
!pip install -q ultralytics

In [ ]:
import ultralytics, torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU 모드")

**로컬 PC에서 실행할 때**: `pip install ultralytics` 후 같은 코드를 쓰면 됩니다. GPU가 없으면 8장에서 `EPOCHS = 10`, `IMGSZ = 416` 정도로 줄이세요. (CPU에서는 한 에포크에 수 분이 걸립니다.)

## 2장. 데이터셋 내려받기 — YAML 한 장과 폴더 규칙

Ultralytics가 제공하는 데이터셋은 YAML 이름만 넣으면 자동으로 내려받습니다. 우리 데이터를 쓸 때도 이 폴더 규칙과 YAML 형식을 그대로 따르면 됩니다.

In [ ]:
from ultralytics.data.utils import check_det_dataset
from pathlib import Path

data = check_det_dataset("medical-pills.yaml")   # 처음 실행 시 약 8MB 다운로드
ROOT = Path(data["train"]).parent.parent
print("데이터 루트:", ROOT)
print("클래스:", data["names"])

In [ ]:
# 실제 YAML 파일 내용 보기
import ultralytics, os
yaml_path = Path(ultralytics.__file__).parent / "cfg" / "datasets" / "medical-pills.yaml"
print(yaml_path.read_text())

In [ ]:
# 폴더 규칙: images/ 와 labels/ 가 같은 이름으로 짝을 이룹니다.
for split in ["train", "val"]:
    imgs = sorted((ROOT / "images" / split).glob("*"))
    lbls = sorted((ROOT / "labels" / split).glob("*.txt"))
    print(f"{split:5s} 이미지 {len(imgs):3d}장 · 라벨 {len(lbls):3d}개 · 예: {imgs[0].name} ↔ {lbls[0].name}")

## 3장. 라벨 한 줄 읽기 — 숫자 다섯 개

`클래스번호 중심x 중심y 너비 높이` 다섯 숫자이고, 좌표는 모두 **이미지 크기로 나눈 비율(0~1)**입니다. 그래서 이미지를 줄이거나 키워도 라벨을 그대로 쓸 수 있습니다.

In [ ]:
from PIL import Image

lbl = ROOT / "labels" / "val" / "Frame_144.txt"
img = ROOT / "images" / "val" / "Frame_144.jpg"
W, H = Image.open(img).size
first = lbl.read_text().splitlines()[0]
print("첫 줄:", first)
cls, cx, cy, w, h = first.split()
cx, cy, w, h = map(float, (cx, cy, w, h))
print(f"이미지 크기 {W}×{H}")
print(f"중심 ({cx*W:.0f}, {cy*H:.0f}) px · 크기 {w*W:.0f}×{h*H:.0f} px")

In [ ]:
# 흔한 실수: 중심점을 왼쪽 위 모서리로 잘못 읽으면 박스가 어디로 밀리는지 직접 봅니다.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

im = Image.open(img)
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(im)
# 올바른 해석(초록): 중심 → 왼쪽 위 = 중심 - 크기/2
ax.add_patch(patches.Rectangle(((cx - w/2)*W, (cy - h/2)*H), w*W, h*H, fill=False, color="lime", lw=3, label="correct (center)"))
# 잘못된 해석(빨강): cx, cy를 왼쪽 위 모서리로 사용
ax.add_patch(patches.Rectangle((cx*W, cy*H), w*W, h*H, fill=False, color="red", lw=3, ls="--", label="wrong (corner)"))
ax.legend(); ax.set_title("Frame_144 — first label line"); ax.axis("off"); plt.show()

### 라벨 통계 뽑기
현업에서 데이터를 받으면 가장 먼저 하는 일입니다. 이미지당 물체 수, 박스 크기, 입력 크기(640) 기준으로 몇 픽셀인지를 봅니다.

In [ ]:
import numpy as np

def read_labels(split):
    rows = []
    for p in sorted((ROOT / "labels" / split).glob("*.txt")):
        for line in p.read_text().splitlines():
            if line.strip():
                c, cx, cy, w, h = line.split()
                rows.append((p.stem, int(c), float(cx), float(cy), float(w), float(h)))
    return rows

train = read_labels("train")
n_img = len(set(r[0] for r in train))
w_arr = np.array([r[4] for r in train]); h_arr = np.array([r[5] for r in train])
print(f"학습 세트 이미지 {n_img}장 · 박스 {len(train)}개 · 이미지당 평균 {len(train)/n_img:.1f}개")
print(f"박스 너비 평균 {w_arr.mean():.3f} · 높이 평균 {h_arr.mean():.3f} (이미지 한 변 대비 비율)")
print(f"박스 한 변 평균 ≈ {(w_arr.mean()+h_arr.mean())/2*100:.0f}% of image side")

In [ ]:
# 입력 크기 640 기준 박스 크기(픽셀) 분포. 8px(격자 한 칸)보다 작은 박스가 얼마나 되는지 봅니다.
IMGSZ_REF = 640
px = np.minimum(w_arr, h_arr) * IMGSZ_REF      # 짧은 변 기준
plt.figure(figsize=(8, 4))
plt.hist(px, bins=30, color="#E0602D")
plt.axvline(8, color="k", ls="--", label="8 px (stride of P3)")
plt.xlabel("box short side at imgsz=640 (px)"); plt.ylabel("count"); plt.legend(); plt.show()
print(f"8px 미만 박스 비율: {(px < 8).mean()*100:.1f}% · 중앙값 {np.median(px):.0f}px")

In [ ]:
# 이 데이터는 어떻게 나뉘어 있을까? 파일명의 프레임 번호를 정렬해 봅니다.
def frame_nums(split):
    return sorted(int(p.stem.split("_")[1]) for p in (ROOT / "labels" / split).glob("*.txt"))

tr, va = frame_nums("train"), frame_nums("val")
print("train 프레임 번호 앞부분:", tr[:15])
print("val   프레임 번호 앞부분:", va[:15])
print("train 프레임 간격 최빈값:", np.bincount(np.diff(tr)).argmax())
# 질문: 검증 세트의 프레임이 학습 세트 프레임 사이사이에 끼어 있다면, 이 검증 점수는 '처음 보는 사진'에 대한 점수일까요?

## 4장. IoU 손계산

IoU = 겹친 넓이 ÷ 합친 넓이. 합친 넓이는 (A + B − 겹친 부분)입니다. "절반 겹치면 0.5"가 아니라는 점을 직접 확인합니다.

In [ ]:
def iou(a, b):
    # a, b = (x1, y1, x2, y2)
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1]); area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)

A = (0, 0, 10, 10)                          # 넓이 100
for shift in (2, 5, 8):
    B = (shift, 0, 10 + shift, 10)          # 오른쪽으로 shift만큼 민 같은 크기 박스
    print(f"{shift}만큼 밀면 겹침 {10*(10-shift):3d} → IoU = {iou(A, B):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, shift in zip(axes, (2, 5, 8)):
    B = (shift, 0, 10 + shift, 10)
    ax.add_patch(patches.Rectangle((0, 0), 10, 10, fill=False, color="#2E5E8C", lw=3))
    ax.add_patch(patches.Rectangle((shift, 0), 10, 10, fill=False, color="#5E666B", lw=3, ls="--"))
    ax.add_patch(patches.Rectangle((shift, 0), 10 - shift, 10, color="#E0602D", alpha=0.6))
    ax.set_xlim(-1, 20); ax.set_ylim(-1, 11); ax.set_aspect("equal"); ax.set_title(f"shift {shift} → IoU {iou(A, B):.2f}")
plt.show()

## 5장. NMS 손계산

모델은 한 물체에 박스를 여러 개 냅니다. NMS는 ① 신뢰도가 가장 높은 박스를 남기고 ② 그 박스와 IoU가 기준값 이상인 박스를 지우고 ③ 남은 박스로 반복합니다.

In [ ]:
def nms(boxes, scores, iou_thr):
    order = sorted(range(len(boxes)), key=lambda i: -scores[i])
    keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < iou_thr]
    return keep

# 장난감 예시: 알약 A와 바로 옆에 붙은 알약 B. 모델이 박스를 다섯 개 냈습니다.
names  = ["A", "A-dup1", "A-dup2", "B(neighbor)", "B-dup"]
boxes  = [(10,10,50,50), (12,12,52,52), (20,10,60,50), (28,10,68,50), (30,12,70,52)]
scores = [0.9, 0.8, 0.7, 0.85, 0.6]
print("A와 B(옆 알약)의 IoU =", round(iou(boxes[0], boxes[3]), 2))
for thr in (0.3, 0.5, 0.7, 0.9):
    keep = nms(boxes, scores, thr)
    print(f"iou_thr={thr}: {len(keep)}개 남음 → {[names[i] for i in keep]}")
# 기준값 0.3: 옆 알약 B까지 지워짐 (너무 낮음) · 0.5: A, B 하나씩 (적절) · 0.7: A의 중복이 남음 · 0.9: 거의 안 지움

In [ ]:
# 실습 데이터에서 '서로 다른 알약'끼리 IoU가 얼마나 되는지 재 봅니다 (정답 박스끼리).
from itertools import combinations
from collections import defaultdict

by_img = defaultdict(list)
for stem, c, cx, cy, w, h in train:
    by_img[stem].append((cx - w/2, cy - h/2, cx + w/2, cy + h/2))

max_ious = []
for stem, bs in by_img.items():
    best = max((iou(a, b) for a, b in combinations(bs, 2)), default=0)
    max_ious.append(best)
print(f"이미지별 '서로 다른 알약 간 최대 IoU'의 평균 {np.mean(max_ious):.3f} · 최대 {np.max(max_ious):.3f}")
# 질문: 이 값이 NMS 기본 기준값 0.7보다 훨씬 작다면, 이 데이터에서 NMS가 옆 알약을 지울 위험은 얼마나 될까요?

## 6장. AP 손계산

한 클래스 안에서 예측을 신뢰도 순으로 줄 세우고, **진짜를 새로 찾을 때마다 그 시점의 정밀도를 기록해 평균**합니다. 못 찾은 정답은 0으로 들어갑니다.

In [ ]:
def ap_simple(is_true, n_gt):
    tp = 0; precs = []
    for k, t in enumerate(is_true, start=1):
        if t:
            tp += 1; precs.append(tp / k)
    precs += [0.0] * (n_gt - tp)            # 못 찾은 정답은 0점
    return sum(precs) / n_gt

print("슬라이드 예시 [맞음,틀림,맞음,틀림,맞음], 정답 3개 →", round(ap_simple([1,0,1,0,1], 3), 2))
print("1등이 틀리면   [틀림,맞음,맞음,틀림,맞음], 정답 3개 →", round(ap_simple([0,1,1,0,1], 3), 2))
print("하나를 못 찾음 [맞음,틀림,맞음],           정답 3개 →", round(ap_simple([1,0,1], 3), 2))

In [ ]:
# Ultralytics가 쓰는 방식(COCO식): PR 곡선을 단조 감소로 다듬고 101개 점에서 보간해 면적을 구합니다.
def ap_coco101(is_true, n_gt):
    tp = np.cumsum(is_true); fp = np.cumsum(1 - np.array(is_true))
    recall = tp / n_gt; precision = tp / (tp + fp)
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[1.0], precision, [0.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))   # 오른쪽부터 최대값으로 덮기
    x = np.linspace(0, 1, 101)
    trapz = getattr(np, "trapezoid", None) or np.trapz      # NumPy 2.0에서 이름이 바뀜
    return trapz(np.interp(x, mrec, mpre), x)

for seq in ([1,0,1,0,1], [0,1,1,0,1]):
    print(seq, "단순 평균", round(ap_simple(seq, 3), 3), "· COCO 101점 보간", round(ap_coco101(seq, 3), 3))

## 7장. 사전학습 모델을 그대로 돌려 보기

COCO 80개 클래스로 학습된 `yolo26n.pt`는 알약이라는 클래스를 모릅니다. 아는 80개 안에서만 답할 수 있으니 무엇으로 착각하는지 봅니다.

In [ ]:
from ultralytics import YOLO

coco = YOLO("yolo26n.pt")                        # 처음 실행 시 가중치 다운로드
sample = sorted((ROOT / "images" / "val").glob("*.jpg"))[0]
r = coco.predict(sample, conf=0.25, nms=True, verbose=False)[0]
found = [(coco.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)]
print("찾았다고 한 것:", found if found else "아무것도 못 찾음")
Image.fromarray(r.plot()[..., ::-1]).resize((960, 540))

## 8장. 전이학습 — 눈썰미 좋은 모델에 알약만 새로 가르치기

같은 `yolo26n.pt`에서 출발해 알약 92장으로 추가 학습합니다. 기본값에서 **이유가 있는 몇 줄만** 바꿉니다.

| 바꾼 것 | 이유 |
|---|---|
| `epochs=30, patience=10` | 수업 시간 안에 끝내기. 현업은 100~300 |
| `fliplr=0.0` | 알약 각인(글자)은 좌우로 뒤집히지 않음 |
| `hsv_v=0.6` | 응급실·휴대폰 촬영은 밝기가 제각각 (기본 0.4) |

In [ ]:
# 증강 기본값 확인 — 어떤 변화가 '자동으로' 들어가는지 알고 시작합니다.
from ultralytics.cfg import get_cfg
cfg = get_cfg()
for k in ["hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "fliplr", "flipud", "mosaic", "close_mosaic"]:
    print(f"{k:12s} = {getattr(cfg, k)}")

In [ ]:
EPOCHS = 30
IMGSZ = 640

model = YOLO("yolo26n.pt")
results = model.train(
    data="medical-pills.yaml",
    epochs=EPOCHS, patience=10,
    imgsz=IMGSZ, batch=16,
    fliplr=0.0,
    hsv_v=0.6,
    seed=0,
    project="runs_pill", name="yolo26n", exist_ok=True,
)
RUN = Path(results.save_dir)
print("결과 폴더:", RUN)

In [ ]:
# 학습에 실제로 들어간 배치: 4장을 이어 붙인 모자이크와 잘린 알약이 보입니다.
Image.open(RUN / "train_batch0.jpg").resize((960, 960))

In [ ]:
# (비교 실험용) YOLO11로 같은 실험을 하려면 파일 이름만 바꾸면 됩니다.
# model11 = YOLO("yolo11n.pt")
# model11.train(data="medical-pills.yaml", epochs=EPOCHS, imgsz=IMGSZ, fliplr=0.0, hsv_v=0.6, seed=0, project="runs_pill", name="yolo11n", exist_ok=True)

## 9장. 학습 곡선 읽기

학습 손실만 보지 말고 검증 손실을 같이 봅니다. 학습은 내려가는데 검증이 올라가면 학습 사진을 외우는 중입니다. YOLO26은 DFL 대신 `l1_loss`가 찍힙니다.

In [ ]:
import pandas as pd
df = pd.read_csv(RUN / "results.csv")
df.columns = [c.strip() for c in df.columns]
print(list(df.columns))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes[:3], ["box_loss", "cls_loss", "l1_loss"]):
    ax.plot(df["epoch"], df[f"train/{name}"], label="train", color="#2E5E8C")
    ax.plot(df["epoch"], df[f"val/{name}"], label="val", color="#E0602D")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.legend()
axes[3].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50", color="#5E666B")
axes[3].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95", color="#E0602D")
axes[3].set_title("mAP (val)"); axes[3].set_xlabel("epoch"); axes[3].legend()
plt.tight_layout(); plt.show()

## 10장. 평가 — mAP50과 mAP50-95

`val()`은 신뢰도 0.001부터 모든 예측을 모아 PR 곡선 전체를 그립니다. mAP50은 IoU 0.5 이상이면 정답으로 치는 너그러운 기준, mAP50-95는 0.50~0.95 열 단계의 평균입니다.

In [ ]:
best = YOLO(RUN / "weights" / "best.pt")
m = best.val(data="medical-pills.yaml", imgsz=IMGSZ, nms=True, plots=True, verbose=False)
print(f"precision {m.box.mp:.3f} · recall {m.box.mr:.3f}")
print(f"mAP50 {m.box.map50:.3f} · mAP50-95 {m.box.map:.3f}")
# 질문: 두 mAP의 차이가 크다면 '찾긴 했는데 박스가 정확히 맞지는 않는다'는 뜻입니다. 우리 결과는 어떤가요?

## 11장. 추론 — 신뢰도 기준은 서비스가 정한다

같은 모델이라도 `conf`를 어디에 두느냐에 따라 "찾은 개수"가 달라집니다. 정답 개수와 비교해 봅니다.

In [ ]:
val_imgs = sorted((ROOT / "images" / "val").glob("*.jpg"))
gt_count = {p.stem: len([l for l in (ROOT / "labels" / "val" / f"{p.stem}.txt").read_text().splitlines() if l.strip()]) for p in val_imgs}
total_gt = sum(gt_count.values())

for conf in (0.1, 0.25, 0.5, 0.75):
    rs = best.predict(val_imgs, conf=conf, nms=True, imgsz=IMGSZ, verbose=False)
    n_pred = sum(len(r.boxes) for r in rs)
    print(f"conf={conf:<5} 탐지 {n_pred:4d}개 · 정답 {total_gt}개 · 비율 {n_pred/total_gt:.2f}")
# 질문: 응급실 약봉지라면 '놓치는 실수'와 '없는 걸 찾는 실수' 중 어느 쪽이 더 비쌀까요?

In [ ]:
# 두 개의 출력 머리: nms=True(one-to-many + NMS, 기본값) vs nms=False(one-to-one, NMS 없음)
# nms 인자는 매 호출마다 직접 적어 줍니다.
for flag in (True, False):
    rs = best.predict(val_imgs, conf=0.25, nms=flag, imgsz=IMGSZ, verbose=False)
    n_pred = sum(len(r.boxes) for r in rs)
    mean_conf = float(np.mean([float(c) for r in rs for c in r.boxes.conf])) if n_pred else 0
    print(f"nms={flag!s:5}: 탐지 {n_pred:4d}개 · 평균 신뢰도 {mean_conf:.2f}")
# 짧게 학습한 모델에서는 one-to-one 쪽이 더 적게 찾거나 신뢰도가 낮게 나올 수 있습니다.

In [ ]:
# 주의: nms 설정은 같은 모델 객체의 다음 호출에도 남습니다. 인자를 생략하면 '직전 설정'이 그대로 쓰입니다.
few = val_imgs[:3]
n = lambda rs: sum(len(r.boxes) for r in rs)
print("nms=False 로 호출      :", n(best.predict(few, conf=0.25, nms=False, imgsz=IMGSZ, verbose=False)))
print("이어서 인자 없이 호출   :", n(best.predict(few, conf=0.25, imgsz=IMGSZ, verbose=False)), "← nms=False가 유지됨")
print("nms=True 로 되돌림     :", n(best.predict(few, conf=0.25, nms=True, imgsz=IMGSZ, verbose=False)))
# 결과가 이상하면 nms 설정부터 의심하고, 매 호출마다 직접 적어 주는 습관을 들이세요.

In [ ]:
# 한 장의 결과를 표로: 박스 좌표 · 클래스 · 신뢰도
r = best.predict(val_imgs[0], conf=0.25, nms=True, imgsz=IMGSZ, verbose=False)[0]
tab = pd.DataFrame(r.boxes.xyxy.cpu().numpy().round(0), columns=["x1", "y1", "x2", "y2"])
tab["cls"] = [best.names[int(c)] for c in r.boxes.cls]
tab["conf"] = r.boxes.conf.cpu().numpy().round(2)
print(f"{val_imgs[0].name}: 정답 {gt_count[val_imgs[0].stem]}개, 탐지 {len(tab)}개")
tab.head(10)

## 12장. 오류 분석 — mAP 한 숫자보다 어디서 틀리는지

예측과 정답을 IoU 0.5 기준으로 짝지어 사진별 TP(맞춤)·FP(헛탐지)·FN(놓침)을 세고, 가장 많이 틀린 사진을 열어 봅니다.

In [ ]:
def match(pred_boxes, pred_conf, gt_boxes, thr=0.5):
    order = np.argsort(-np.array(pred_conf)) if len(pred_conf) else []
    used = set(); tp = 0
    for i in order:
        best_j, best_iou = -1, thr
        for j, g in enumerate(gt_boxes):
            if j in used: continue
            v = iou(pred_boxes[i], g)
            if v >= best_iou: best_j, best_iou = j, v
        if best_j >= 0: used.add(best_j); tp += 1
    return tp, len(pred_boxes) - tp, len(gt_boxes) - tp   # TP, FP, FN

def gt_xyxy(stem, W, H):
    out = []
    for line in (ROOT / "labels" / "val" / f"{stem}.txt").read_text().splitlines():
        if line.strip():
            _, cx, cy, w, h = map(float, line.split())
            out.append(((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H))
    return out

rows = []
preds = best.predict(val_imgs, conf=0.25, nms=True, imgsz=IMGSZ, verbose=False)
for p, r in zip(val_imgs, preds):
    W, H = r.orig_shape[1], r.orig_shape[0]
    pb = r.boxes.xyxy.cpu().numpy().tolist(); pc = r.boxes.conf.cpu().numpy().tolist()
    tp, fp, fn = match(pb, pc, gt_xyxy(p.stem, W, H))
    rows.append((p.stem, len(pb), tp, fp, fn))
err = pd.DataFrame(rows, columns=["image", "pred", "TP", "FP", "FN"]).sort_values(["FP", "FN"], ascending=False)
print("전체 TP/FP/FN:", err[["TP", "FP", "FN"]].sum().to_dict())
err.head(8)

In [ ]:
# 가장 많이 틀린 사진 두 장: 초록 = 정답, 빨강 = 예측. 초록 없이 빨강만 있으면 FP, 초록만 있으면 FN.
worst = err.head(2)["image"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, stem in zip(axes, worst):
    p = ROOT / "images" / "val" / f"{stem}.jpg"
    im = Image.open(p); W, H = im.size
    ax.imshow(im)
    for x1, y1, x2, y2 in gt_xyxy(stem, W, H):
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="lime", lw=2))
    r = best.predict(p, conf=0.25, nms=True, imgsz=IMGSZ, verbose=False)[0]
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf):
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="red", lw=2, ls="--"))
        ax.text(x1, y1-4, f"{float(c):.2f}", color="red", fontsize=8)
    ax.set_title(f"{stem} — {err.set_index('image').loc[stem].to_dict()}"); ax.axis("off")
plt.show()
# 질문: 빨간 박스만 있는 곳을 보세요. 모델이 틀린 걸까요, 라벨이 빠진 걸까요? (예: 사진 가장자리에 잘린 알약)

## 13장. 확장 — 모델 혼자 맞히지 말고 처방 세트로 보정하기

실습 데이터는 클래스가 하나뿐이라 "무슨 약인가"는 못 배웁니다. 그래서 이 장은 **탐지 결과가 아래처럼 나왔다고 가정한 가상 데이터**로 보정 규칙만 연습합니다.

1. 확실한 알약으로 가능한 처방 세트를 좁힌다
2. 애매한 알약은 좁혀진 세트 안에서 고른다
3. 완성된 세트를 질환과 연결한다

> 알약 코드·세트·질환은 흐름을 보여 주기 위한 예시이며 실제 처방 정보가 아닙니다. 어느 단계든 결과는 참고 정보이고 최종 판단은 의료진이 합니다.

In [ ]:
# 가상의 탐지 결과: 알약마다 상위 후보와 확률 (YOLO 기본 출력은 박스당 클래스 하나이므로,
# 실제로는 원시 출력을 읽거나 탐지 후 별도 분류 모델로 확률 분포를 받아야 합니다.)
detections = [
    {"P_A": 0.94, "P_C": 0.03},
    {"P_B": 0.91, "P_F": 0.05},
    {"P_D": 0.88, "P_H": 0.07},
    {"P_G": 0.48, "P_E": 0.45},          # 애매한 흰 알약
]
# 가상의 처방 세트 사전
RX_SETS = {
    "SET_01": {"pills": {"P_A", "P_C", "P_F"},      "condition": "고혈압"},
    "SET_02": {"pills": {"P_B", "P_G", "P_H"},      "condition": "당뇨"},
    "SET_03": {"pills": {"P_A", "P_B", "P_D", "P_E"}, "condition": "고혈압 + 당뇨"},
}
SURE = 0.80   # 이 값 이상이면 '확실한 알약'

def correct_with_sets(detections, rx_sets, sure=SURE):
    sure_pills = {max(d, key=d.get) for d in detections if max(d.values()) >= sure}
    # 1) 확실한 알약을 모두 포함하는 세트만 남긴다
    candidates = {k: v for k, v in rx_sets.items() if sure_pills <= v["pills"]}
    allowed = set().union(*(v["pills"] for v in candidates.values())) if candidates else set()
    # 2) 애매한 알약은 후보 세트 안의 알약 중에서 고른다
    final = []
    for d in detections:
        top = max(d, key=d.get)
        if d[top] >= sure or not allowed:
            final.append(top); continue
        inset = {k: v for k, v in d.items() if k in allowed}
        final.append(max(inset, key=inset.get) if inset else top)
    # 3) 완성된 세트를 질환과 연결
    matched = [k for k, v in candidates.items() if set(final) <= v["pills"]]
    return sure_pills, candidates.keys(), final, [(k, rx_sets[k]["condition"]) for k in matched]

sure_pills, cands, final, cond = correct_with_sets(detections, RX_SETS)
print("확실한 알약:", sure_pills)
print("후보 세트:", list(cands))
print("보정 후 알약:", final)
print("추정 질환(참고 정보):", cond)

In [ ]:
# 직접 바꿔 보기: SURE를 0.95로 올리면? 애매한 알약의 확률을 P_G 0.60 / P_E 0.30으로 바꾸면? 세트를 하나 더 추가하면?
detections[3] = {"P_G": 0.60, "P_E": 0.30}
print(correct_with_sets(detections, RX_SETS, sure=0.80)[2:])

## 14장. 과제 — 내 사진 20장으로 직접 해 보기

1. 휴대폰으로 작은 물체(열쇠, 지우개, 동전 등)를 **20장 이상** 찍습니다. 파일 이름은 영문·숫자만 (`key_001.jpg`).
2. 라벨링 도구(CVAT, Label Studio, Roboflow, LabelImg 중 하나)에서 박스를 그리고 **YOLO 형식**으로 내보냅니다.
3. 아래 폴더 규칙대로 놓고 YAML을 만든 뒤, 8장 코드에서 `data=`만 바꿔 학습합니다.
4. 슬라이드 31의 **라벨 기준서 표**를 자기 데이터로 채워 옵니다.

```
my_data/
├ images/train/  key_001.jpg …
├ images/val/    key_017.jpg …
├ labels/train/  key_001.txt …
└ labels/val/    key_017.txt …
```

In [ ]:
# 1) YAML 템플릿 — 경로와 클래스 이름만 바꾸면 됩니다.
my_yaml = '''
path: /content/my_data      # 데이터 루트 (Colab 기준)
train: images/train
val: images/val
names:
  0: key
  1: eraser
'''
Path("my_data.yaml").write_text(my_yaml.strip() + "\n")
print(Path("my_data.yaml").read_text())

In [ ]:
# 2) 학습 — 데이터를 올린 뒤 주석을 풀고 실행합니다.
# model = YOLO("yolo26n.pt")
# model.train(data="my_data.yaml", epochs=30, imgsz=640, seed=0, project="runs_mine", name="first", exist_ok=True)

In [ ]:
# 3) 2회차를 위해 오늘 학습 결과를 Google Drive에 저장해 둡니다 (Colab 전용).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dst = Path("/content/drive/MyDrive/yolo_bootcamp/day1_pill"); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(RUN / "weights" / "best.pt", dst / "best.pt")
    shutil.copy(RUN / "results.csv", dst / "results.csv")
    print("저장 완료:", dst)
except ImportError:
    print(f"Colab이 아닙니다. {RUN / 'weights' / 'best.pt'} 를 직접 보관하세요.")

## 오늘 정리

- 모델을 바꾸기 전에 **데이터부터** 의심한다 (3장 프레임 번호, 12장 라벨 누락)
- 라벨 한 줄은 숫자 다섯 개, 좌표는 **중심점 + 비율**
- IoU·NMS·AP는 손으로 계산할 수 있다 — 절반 겹치면 0.33
- 신뢰도 기준(`conf`)은 모델이 아니라 **서비스**가 정한다
- 오류 분석은 mAP 한 숫자보다 **어디서 틀리는지**
- 도메인 지식(처방 세트)으로 결과를 보정한다

다음 시간(2회차): 혼동 행렬, PR 곡선으로 임계값 정하기, 데이터 추가 수집 전략.